# 02 - Feature Engineering · DataCo Smart Supply Chain

Construcción y validación de variables para predecir `Late_delivery_risk`.

## Objetivos

1. Crear variables temporales, económicas, de retraso y de riesgo histórico.
2. **Evitar data leakage** mediante split temporal y target encoding entrenado solo en train.
3. Codificar variables categóricas.
4. Visualizar el comportamiento de las nuevas features.
5. Calcular una **importancia preliminar de variables** con Random Forest.
6. Persistir los datasets listos para ML en `data/processed/`.

## Variables creadas

| Variable | Tipo | Uso |
|----------|------|-----|
| `delay_days`, `is_late` | ex-post | EDA / target alternativo |
| `order_month`, `order_dayofweek`, `order_year` | ex-ante | modelo |
| `profit_margin`, `discount_pct` | ex-ante | modelo |
| `shipping_efficiency` | ex-post | EDA |
| `city_risk_score`, `route_risk_score`, `customer_delay_rate` | ex-ante (entrenadas en train) | modelo |

## 0. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import configurar_logger, guardar_dataset
from src.preprocessing import limpiar_dataco
from src.features import (
    variables_temporales,
    variables_retraso,
    variables_economicas,
    crear_risk_scores,
    codificar_categoricas,
    split_temporal,
    eliminar_features_leakage,
    seleccionar_X_y,
    construir_features_dataco,
    TARGET,
)

configurar_logger('feature_eng')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Cargar el dataset limpio

Usa `data/processed/dataco_clean.parquet` generado en `01_eda_dataco.ipynb`.
Si no existe, lo construye desde cero llamando a `limpiar_dataco()`.

In [ ]:
ruta_clean = Path('../data/processed/dataco_clean.parquet')

if ruta_clean.exists():
    df = pd.read_parquet(ruta_clean)
    print(f'Cargado: {df.shape}')
else:
    print('No existe dataco_clean.parquet, generándolo desde raw...')
    df = limpiar_dataco(guardar=True)

df.head()

## 2. Variables temporales

Generamos las features de calendario a partir de `order_date_dateorders`.

In [ ]:
df = variables_temporales(df, columna_fecha='order_date_dateorders')
df[['order_year', 'order_month', 'order_dayofweek', 'order_quarter', 'order_is_weekend']].head()

In [ ]:
# Visualización: tasa de retraso por mes y día de la semana
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

(df.groupby('order_month')[TARGET].mean()
   .plot(kind='bar', ax=axes[0], color='#4c8acc'))
axes[0].set_title('Tasa de retraso por mes')
axes[0].set_ylabel('P(late_delivery_risk = 1)')
axes[0].set_xlabel('Mes')

(df.groupby('order_dayofweek')[TARGET].mean()
   .plot(kind='bar', ax=axes[1], color='#cc4c4c'))
axes[1].set_title('Tasa de retraso por día de la semana')
axes[1].set_xticklabels(['Lun', 'Mar', 'Mie', 'Jue', 'Vie', 'Sab', 'Dom'], rotation=0)
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

## 3. Variables de retraso (`delay_days`, `is_late`)

⚠️ **Ex-post**: sirven para EDA y como target alternativo. NO se incluyen en el modelo.

In [ ]:
df = variables_retraso(df)
df[['days_for_shipment_scheduled', 'days_for_shipping_real', 'delay_days', 'is_late', TARGET]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['delay_days'], bins=range(df['delay_days'].min(), df['delay_days'].max()+2),
             ax=axes[0], color='#4c8acc')
axes[0].axvline(0, color='red', linestyle='--', label='Sin retraso')
axes[0].set_title('Distribución de delay_days')
axes[0].set_xlabel('Días de retraso (real - scheduled)')
axes[0].legend()

consistencia = pd.crosstab(df['is_late'], df[TARGET])
sns.heatmap(consistencia, annot=True, fmt=',', cmap='Blues', ax=axes[1])
axes[1].set_title('is_late (derivado) vs late_delivery_risk (target)')
axes[1].set_xlabel('late_delivery_risk')
axes[1].set_ylabel('is_late')
plt.tight_layout()
plt.show()

## 4. Variables económicas

In [ ]:
df = variables_economicas(df)
df[['profit_margin', 'shipping_efficiency', 'discount_pct']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['profit_margin'], bins=50, ax=axes[0], color='#4c8acc')
axes[0].set_title('Distribución de profit_margin')
axes[0].set_xlabel('Margen sobre ventas')

sns.boxplot(data=df, x=TARGET, y='profit_margin', ax=axes[1])
axes[1].set_title('profit_margin por clase')
axes[1].set_xlabel('late_delivery_risk')
plt.tight_layout()
plt.show()

## 5. Split temporal (anti-leakage)

Dividimos cronológicamente **antes** de calcular target encoding.

In [ ]:
df_train, df_test = split_temporal(df, columna_fecha='order_date_dateorders', test_size=0.2)
print(f'Tasa de retraso en train: {df_train[TARGET].mean():.2%}')
print(f'Tasa de retraso en test:  {df_test[TARGET].mean():.2%}')

## 6. Risk scores (target encoding suavizado)

Calculamos:
- `city_risk_score` (por `order_city`)
- `route_risk_score` (combinación `market` × `order_region`)
- `customer_delay_rate` (por `customer_id`)

El encoder se entrena **solo en train** y se aplica al test → cero leakage.

In [ ]:
df_train, df_test, encoder = crear_risk_scores(
    df_train, df_test, target=TARGET, m=30
)
print(f'Media global de retraso (train): {encoder.media_global_:.4f}')

cols_score = ['city_risk_score', 'route_risk_score', 'customer_delay_rate']
cols_score = [c for c in cols_score if c in df_train.columns]
df_train[cols_score].describe()

In [ ]:
# Visualización de los risk scores en train
fig, axes = plt.subplots(1, len(cols_score), figsize=(5 * len(cols_score), 4))
if len(cols_score) == 1:
    axes = [axes]

for ax, col in zip(axes, cols_score):
    sns.histplot(df_train[col], bins=40, ax=ax, color='#4c8acc')
    ax.axvline(encoder.media_global_, color='red', linestyle='--', label='Media global')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

### Top ciudades de mayor riesgo (train)

In [ ]:
if 'order_city' in df_train.columns:
    top_ciudades = (
        df_train.groupby('order_city')
        .agg(ordenes=(TARGET, 'count'), riesgo=(TARGET, 'mean'))
        .query('ordenes >= 50')
        .sort_values('riesgo', ascending=False)
        .head(15)
    )
    display(top_ciudades)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    top_ciudades['riesgo'].plot(kind='barh', ax=ax, color='#cc4c4c')
    ax.set_title('Top 15 ciudades con mayor tasa de retraso (>= 50 órdenes)')
    ax.set_xlabel('Tasa de retraso')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 7. Codificación de categóricas y limpieza final

In [ ]:
categoricas = [
    c for c in ['shipping_mode', 'customer_segment', 'market', 'order_region',
                'category_name', 'department_name']
    if c in df_train.columns
]
print(f'Categóricas a codificar: {categoricas}')

df_train_enc = codificar_categoricas(df_train, categoricas, metodo='onehot')
df_test_enc = codificar_categoricas(df_test, categoricas, metodo='onehot')

df_train_enc = eliminar_features_leakage(df_train_enc)
df_test_enc = eliminar_features_leakage(df_test_enc)

# Alinear columnas
cols_comunes = sorted(set(df_train_enc.columns) | set(df_test_enc.columns))
for col in cols_comunes:
    if col not in df_train_enc.columns:
        df_train_enc[col] = 0
    if col not in df_test_enc.columns:
        df_test_enc[col] = 0
df_train_enc = df_train_enc[cols_comunes]
df_test_enc = df_test_enc[cols_comunes]

X_train, y_train = seleccionar_X_y(df_train_enc)
X_test, y_test = seleccionar_X_y(df_test_enc)

print(f'\nX_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features finales: {X_train.shape[1]}')

## 8. Correlación de las nuevas features con el target

In [ ]:
df_corr = X_train.copy()
df_corr['target'] = y_train.values
corr_target = df_corr.corr()['target'].drop('target')

top_corr = corr_target.abs().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
top_corr.plot(kind='barh', ax=ax, color='#4c8acc')
ax.set_title('Top 20 features por correlación absoluta con late_delivery_risk')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

corr_target.reindex(top_corr.index).round(4).to_frame('corr')

## 9. Importancia preliminar con Random Forest

Modelo rápido para evaluar señal de las features. **No es el modelo final** — el entrenamiento riguroso con XGBoost y MLflow está en `03_training_mlflow.ipynb`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced',
)
rf.fit(X_train, y_train)

proba_test = rf.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba_test)
print(f'ROC-AUC (test): {auc:.4f}\n')
print(classification_report(y_test, pred_test, digits=4))

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
top_imp = importancias.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_imp.plot(kind='barh', ax=ax, color='#cc4c4c')
ax.set_title('Top 20 features más importantes (Random Forest)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

top_imp.round(4).to_frame('importancia')

## 10. Pipeline orquestador (one-shot)

Todo lo anterior en una sola llamada — útil para reproducibilidad.

In [ ]:
# Volvemos a cargar el dataset limpio para correr el pipeline desde cero
df_fresh = pd.read_parquet('../data/processed/dataco_clean.parquet')

resultado = construir_features_dataco(
    df_fresh,
    metodo_categoricas='onehot',
    test_size=0.2,
    m_suavizado=30,
)

print(f"X_train: {resultado['X_train'].shape}")
print(f"X_test:  {resultado['X_test'].shape}")
print(f"Features finales ({len(resultado['features_finales'])}):\n")
print(resultado['features_finales'][:25], '...')

## 11. Persistir los datasets de modelado

In [ ]:
out_train = resultado['X_train'].copy()
out_train[TARGET] = resultado['y_train'].values

out_test = resultado['X_test'].copy()
out_test[TARGET] = resultado['y_test'].values

guardar_dataset(out_train, 'dataco_train.parquet', capa='processed')
guardar_dataset(out_test, 'dataco_test.parquet', capa='processed')

## 12. Conclusiones

- **Anti-leakage**: el target encoding se entrenó solo en train y el split fue temporal.
- **Variables más informativas**: típicamente `customer_delay_rate`, `route_risk_score`, `shipping_mode_*` y `days_for_shipment_scheduled`.
- **Próximo paso**: `03_training_mlflow.ipynb` — entrenamiento riguroso con XGBoost, tracking en MLflow y comparación contra el baseline de Random Forest mostrado aquí.